# Experiments 1–3 | Orientation reliability of pretrained score geometry

**Self-contained, Colab-ready, primarily CPU-friendly. This is geometry research only—not adversarial attacks.**

**Scientific hypotheses** (must be testable against negative outcomes):

1. **Failure mode:** accurate score-Hessian spectral dimension does not imply accurate tangent orientation.
2. **Existing uncertainty:** observable proxies have limited, measurable value for predicting actual tangent failures.
3. **Proposed diagnostic:** cross-scale score-denoiser response provides *incremental* information about tangent orientation that existing proxies lack.

**Experimental integrity:**
- First use *previously inspected* synthetic CSV data for **exploratory** Experiments 1–2. No existing test data are called untouched.
- Reuse original frozen score networks and the **original 20 FIT / 20 CAL predictor** without changing its algorithm.
- Experiment 3 creates **fresh independent sample points** with fixed seeds, then evaluates frozen predictors and a predeclared ridge ablation. These are new points, **not** new independently trained score models.
- True tangent bases and their ranks are used **only to calculate synthetic evaluation labels** (and to reproduce the previous synthetic neighbor-chord proxy; this proxy uses privileged manifold parametrization, see caveats). The *new cross-scale response diagnostic* does **not** use them.
- We report both **oracle-rank tangent error** (isolates orientation) and **estimated-rank tangent error** (closer to a deployable estimator). Results using oracle rank are never presented as deployment performance.
- There is **no assumption** that the new diagnostic works. If the incremental improvement is absent, report that result.

**Outputs:** figures `Fig1_*.png`, `Fig2_*.png`, `Fig3_*.png`, analysis CSVs, model provenance, and `PROPOSAL_EVIDENCE.md` in Google Drive. No synthetic numbers are hard-coded as new experimental results.

**Reuses:** `synthetic_eps_network.pt`, `{family}_eps_R8_seed73_steps2500_v1.pt`, previous `*_LHSD_shared_operator.csv` (if available), original `*_per_point_results.csv`, and the original noise-aware selector script embedded below.

## 0 · Runtime, paths, frozen protocol

In [ ]:
# Standard Colab typically has these; this cell works in local Jupyter too.
%pip -q install numpy pandas scipy scikit-learn matplotlib torch

In [ ]:
from pathlib import Path
import os, json, math, random, hashlib, shutil, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from IPython.display import display

SEED = 73
D = 8
N_FIT = 40
N_TEST = 100
LHSD_SOFT_C = 0.5
LHSD_SOFT_P = 12
SIGMAS = [0.030,0.128,0.342]
# Include all four manifolds if your trained checkpoint files exist in DATA_DIR.
REQUESTED_FAMILIES = ['circle','sphere','ellipse','torus']
N_FRESH_AUDIT = 250  # New independent points per available manifold; increase for stronger point-level CIs
FRESH_POINT_SEED = 20261103
FIXED_SIGMA = 0.128
RESPONSE_SCALE = {0.030:0.128, 0.128:0.342, 0.342:0.128}  # predeclared cross-scale test
DIM_GOOD_CUTOFF = 0.05  # descriptive threshold; previously viewed data => exploratory
TANGENT_BAD_CUTOFF = 0.06  # descriptive threshold; test on fresh points as well
SELECTION_COVERAGES = [0.20,0.40,0.60,0.80,1.00]
N_BOOT = 2000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive')
except ImportError:
    DRIVE = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()
DATA_DIR = DRIVE/'synthetic_tangent_ground_truth'
BENCH_DIR = DRIVE/'LHSD_vs_Calibrated_Geometry_Benchmark'
OUT = DRIVE/'Orientation_Reliability_Exp123'
OUT.mkdir(parents=True, exist_ok=True)
assert DATA_DIR.is_dir(), ('Missing synthetic_tangent_ground_truth folder. '
    'Copy the circle/sphere/ellipse/torus CSVs and score checkpoints to '+str(DATA_DIR))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 2))
print('Device:', DEVICE,'\nSynthetic data:',DATA_DIR,'\nOld benchmark:',BENCH_DIR,'\nOutput:',OUT)

## 1 · Restore the **original** noise-aware predictor without modification
The next cell writes the original fitted selector algorithm exactly as used in our previous synthetic experiments. It is reused for a clean predictor baseline and original conformal scale calibration. Its split is **20 FIT and 20 CAL points per family** from the old 40-point fitting pool. We do not modify its hyperparameters or original features.

In [ ]:
%%writefile /content/Noise_Aware_Tangent_Selector.py
"""Geometry scale selection with held-out true tangent errors.

Run:
python Noise_Aware_Tangent_Selector.py --csv all_per_point_results.csv --out results

No score-model training or inference. Uses saved synthetic observations only.
Statistical coverage is marginal for an exchangeable point from the same synthetic
setting (and with all three sigmas observed), not distribution-free on MNIST.
"""
import argparse, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

FEATS = ['sigma','neighbor_proxy','projector_defect','soft_dimension']
LABEL = 'tangent_error'
REQUIRED = {'model','split','index','sigma',LABEL,'neighbor_proxy','projector_defect','soft_dimension'}


def load_data(path):
    df = pd.read_csv(path)
    missing = REQUIRED-set(df.columns)
    if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
    df = df[df['model']=='learned_score'].copy()
    if df.empty: raise ValueError('No rows with model=learned_score')
    if df[list(REQUIRED - {'model','split'})].isna().any().any():
        raise ValueError('Found missing numbers in required columns')
    if df.duplicated(['split','index','sigma']).any():
        raise ValueError('Duplicate point/sigma rows')
    for sp in ['fit','test']:
        chunk=df[df.split==sp]
        if chunk.empty: raise ValueError(f'Missing {sp} split')
        counts=chunk.groupby('index')['sigma'].nunique()
        if counts.nunique()!=1 or counts.iloc[0]!=df.sigma.nunique():
            raise ValueError(f'{sp} must contain the same complete sigma grid per point')
    return df


def create_splits(df,seed=73):
    fit_ids = sorted(df.loc[df.split=='fit','index'].unique())
    # deterministic image-wise split, not row-wise split
    rng = np.random.default_rng(seed+159)
    ordered = rng.permutation(fit_ids)
    ntrain = max(10, int(np.floor(0.5*len(ordered))))
    if len(ordered)-ntrain<9:
        raise ValueError('At least 19 distinct fitting points needed to allow train+conformal splits')
    train_ids=set(ordered[:ntrain]); cal_ids=set(ordered[ntrain:])
    train=df[(df.split=='fit')&df['index'].isin(train_ids)].copy()
    calib=df[(df.split=='fit')&df['index'].isin(cal_ids)].copy()
    test=df[df.split=='test'].copy()
    return train,calib,test


def choose_predictor(train,seed=73):
    candidates = {
        'ridge': make_pipeline(StandardScaler(),Ridge(alpha=3.0)),
        'random_forest': RandomForestRegressor(n_estimators=120,max_depth=5,
                   min_samples_leaf=3,random_state=seed,n_jobs=-1),
    }
    folds=min(5,train['index'].nunique())
    cv=GroupKFold(n_splits=folds)
    res={}
    for name,base in candidates.items():
        losses=[]
        for tr,va in cv.split(train[FEATS],train[LABEL],groups=train['index']):
            m=clone(base);m.fit(train.iloc[tr][FEATS],train.iloc[tr][LABEL]);
            pred=m.predict(train.iloc[va][FEATS]);
            losses.append(np.mean(np.abs(train.iloc[va][LABEL].to_numpy()-pred)))
        res[name]=float(np.mean(losses))
    selected=min(res,key=res.get)
    model=clone(candidates[selected]);model.fit(train[FEATS],train[LABEL])
    return model,selected,res


def add_pred(df,model):
    v=df.copy()
    v['predicted_error']=np.clip(model.predict(v[FEATS]),0,math.sqrt(2))
    return v


def calibrate(train, calib, alpha=.10):
    # sigma-dependent scale comes only from TRAINING residuals.
    per_sigma = train.assign(residual=lambda d:d[LABEL]-d['predicted_error']).groupby('sigma')\
        ['residual'].agg(lambda a:np.sqrt(np.mean(np.square(a))))
    floors={float(s):max(.01,float(v)) for s,v in per_sigma.items()}
    calib=calib.copy()
    calib['uncertainty_scale']=calib['sigma'].map(floors)
    calib['scaled_abs_residual']=np.abs(calib[LABEL]-calib['predicted_error'])/calib['uncertainty_scale']
    # A maximum over scales protects selection of ANY sigma after calibration.
    scores=calib.groupby('index')['scaled_abs_residual'].max().sort_values().to_numpy()
    n=len(scores)
    k=math.ceil((n+1)*(1-alpha))
    q=math.inf if k>n else float(scores[k-1])
    return floors,q,n,calib


def evaluate_selector(test,floors,q,tau):
    test=test.copy()
    test['uncertainty_scale']=test['sigma'].map(floors)
    test['predicted_upper_bound'] = test['predicted_error']+q*test['uncertainty_scale']
    test['simultaneous_bound_covered']=(test[LABEL]<=test['predicted_upper_bound'])
    methods={
        'raw_neighbor_proxy':'neighbor_proxy',
        'raw_projector_defect':'projector_defect',
        'noise_aware_predicted_error':'predicted_error',
        'calibrated_upper_bound':'predicted_upper_bound',
    }
    cols=['index','sigma',LABEL,'predicted_error','predicted_upper_bound',
          'neighbor_proxy','projector_defect','soft_dimension']
    pieces=[]
    for name,key in methods.items():
        z=test.loc[test.groupby('index')[key].idxmin(),cols].copy()
        z['method']=name
        z['accepted']=(z.predicted_upper_bound<=tau) if name=='calibrated_upper_bound' else True
        pieces.append(z)
    oracle=test.loc[test.groupby('index')[LABEL].idxmin(),cols].copy()
    oracle['method']='oracle_true_error_UNAVAILABLE_ON_MNIST';oracle['accepted']=True
    pieces.append(oracle)
    selected=pd.concat(pieces,ignore_index=True)
    summary=[]
    for name,z in selected.groupby('method'):
        acc=z[z.accepted]
        summary.append({
           'method':name,'coverage':float(len(acc)/len(z)),
           'accepted_count':int(len(acc)),
           'mean_true_error_accepted':float(acc[LABEL].mean()) if len(acc) else np.nan,
           'mean_true_error_all_selected':float(z[LABEL].mean()),
           'fraction_accepted_above_tau':float(np.mean(acc[LABEL]>tau)) if len(acc) else np.nan,
           'sigma_counts_accepted':json.dumps({str(k):int(v) for k,v in acc.sigma.value_counts().sort_index().items()}),
        })
    coverage_by_point=test.groupby('index')['simultaneous_bound_covered'].all()
    return selected,pd.DataFrame(summary).sort_values('mean_true_error_all_selected'),float(coverage_by_point.mean())


def pair_bootstrap(selected,alpha=.05,nboot=4000,seed=73):
    grid=selected.pivot(index='index',columns='method',values=LABEL)
    d=grid['raw_neighbor_proxy']-grid['calibrated_upper_bound']
    vals=d.to_numpy()
    rng=np.random.default_rng(seed+716)
    draws=rng.integers(0,len(vals),size=(nboot,len(vals)))
    boot=vals[draws].mean(axis=1)
    return float(vals.mean()), np.quantile(boot,[alpha/2,1-alpha/2]).tolist()


def run(csv,out,alpha=.10,tau=.15,seed=73):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    df=load_data(csv)
    tr,ca,te=create_splits(df,seed)
    model,name,cv=choose_predictor(tr,seed)
    tr=add_pred(tr,model)
    ca=add_pred(ca,model)
    te=add_pred(te,model)
    scales,q,n,ca=calibrate(tr,ca,alpha)
    selected,summary,observed= evaluate_selector(te,scales,q,tau)
    diff,ci=pair_bootstrap(selected,seed=seed)
    tr.to_csv(out/'train_points.csv',index=False)
    ca.to_csv(out/'calibration_points.csv',index=False)
    te.to_csv(out/'test_points_with_predictions.csv',index=False)
    selected.to_csv(out/'test_selected_by_method.csv',index=False)
    summary.to_csv(out/'method_comparison.csv',index=False)
    report={'source_csv':str(csv),'manifold':'circle R8, learned_score',
      'n_train_points':int(tr['index'].nunique()),'n_calib_points':int(ca['index'].nunique()),
      'n_test_points':int(te['index'].nunique()),'predictor':name,
      'train_only_cross_validation_MAE':cv,'sigmas':sorted(df.sigma.unique().tolist()),
      'alpha':alpha,'requested_error_tolerance':tau,'conformal_q':q,
      'sigma_residual_scales':{str(k):v for k,v in scales.items()},
      'observed_test_simultaneous_coverage':observed,
      'mean_paired_true_error_improvement_vs_raw_proxy':diff,
      'paired_bootstrap_CI95':ci,
      'limitation':'Coverage applies marginally to same-population points, NOT conditional on acceptance, NOT MNIST. The model uses true synthetic tangent labels in fitting and calibration.'}
    (out/'report.json').write_text(json.dumps(report,indent=2))
    print('Selection results on untouched test points:')
    print(summary.to_string(index=False))
    print('\n',json.dumps(report,indent=2))
    fig,axes=plt.subplots(1,2,figsize=(12,4.2))
    plot=summary[~summary['method'].str.contains('oracle')].sort_values('mean_true_error_all_selected')
    axes[0].barh(plot.method,plot.mean_true_error_all_selected)
    axes[0].set(xlabel='Mean TRUE tangent-projector error (all selected)',title='Held-out selection quality')
    axes[0].invert_yaxis()
    st=te.groupby('sigma').agg(true_error=(LABEL,'mean'),predicted_error=('predicted_error','mean'))
    axes[1].plot(st.index,st.true_error,marker='o',label='Actual error')
    axes[1].plot(st.index,st.predicted_error,marker='o',label='Predicted error')
    axes[1].set(xlabel='Noise sigma',ylabel='Mean tangent error',title='Scale-aware prediction (test)')
    axes[1].legend();axes[1].grid(alpha=.2)
    fig.tight_layout();fig.savefig(out/'selector_validation.png',dpi=170,bbox_inches='tight');plt.close(fig)
    print('Saved all reports to',out)
    return summary,report

if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--csv',required=True)
    p.add_argument('--out',default='noise_aware_selector_results')
    p.add_argument('--alpha',type=float,default=.1)
    p.add_argument('--tau',type=float,default=.15)
    p.add_argument('--seed',type=int,default=73)
    a=p.parse_args();run(a.csv,a.out,a.alpha,a.tau,a.seed)

In [ ]:
import sys, importlib
sys.path.insert(0, '/content' if Path('/content/Noise_Aware_Tangent_Selector.py').exists() else str(Path.cwd()))
import Noise_Aware_Tangent_Selector as original_selector
importlib.reload(original_selector)
assert original_selector.FEATS == ['sigma','neighbor_proxy','projector_defect','soft_dimension']
print('Loaded original noise-aware selector:', original_selector.FEATS)

## 2 · Restore the **same synthetic score-model architecture and manifold embeddings**
The source is copied from our earlier benchmark, including circle/sphere/ellipse/torus parametrizations and the score network checkpoint convention. All score models are **frozen**. The new diagnostic is based on the *raw* score Jacobian, not an assumption that this learned epsilon network is an exactly conservative EBM.

In [ ]:
from torch import nn
class EpsNetwork(nn.Module):
    def __init__(self,D=8,width=128):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(D+1,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),nn.Linear(width,D))
    def forward(self,y,sigma):
        if not isinstance(sigma,torch.Tensor):
            sigma=torch.full((len(y),1),float(sigma),device=y.device,dtype=y.dtype)
        if sigma.ndim==1: sigma=sigma[:,None]
        return self.net(torch.cat([y,torch.log(sigma)],dim=1))

_rng=np.random.default_rng(SEED)
B,_=np.linalg.qr(_rng.normal(size=(D,2)))
_vec=np.random.default_rng(SEED+101).normal(size=D)
_vec=_vec-B@(B.T@_vec)
_vec/=np.linalg.norm(_vec)
B3=np.column_stack([B,_vec])
R=1.0; A_ELL,B_ELL=1.30,0.70;R_SPHERE=1.0;R_MAJOR,R_MINOR=1.20,0.45
FAMILIES=('ellipse','sphere','torus')
OFFSET=np.array([-0.14,-0.08,0.08,0.14])

In [ ]:
def sample_family_params(family, n, rng):
    if family == 'ellipse':
        return rng.uniform(0, 2*np.pi, size=(n,1))
    if family == 'sphere':
        return np.column_stack([rng.uniform(0,2*np.pi,size=n), rng.uniform(-1,1,size=n)])
    if family == 'torus':
        return rng.uniform(0,2*np.pi,size=(n,2))
    raise ValueError(family)

def family_points_and_tangent(family, params):
    # Return x:(n,D), true orthonormal tangent rows:(n,d,D).
    p = np.asarray(params,dtype=float)
    if family=='ellipse':
        th=p[:,0]; c=np.cos(th); s=np.sin(th)
        local=np.column_stack([A_ELL*c, B_ELL*s])
        x=local@B.T
        v=(-A_ELL*s)[:,None]*B[:,0]+(B_ELL*c)[:,None]*B[:,1]
        return x, (v/np.linalg.norm(v,axis=1,keepdims=True))[:,None,:]
    if family=='sphere':
        th,z=p[:,0],p[:,1]
        a=np.sqrt(np.clip(1-z*z,1e-12,None))
        local=R_SPHERE*np.column_stack([a*np.cos(th),a*np.sin(th),z])
        t1=np.column_stack([-np.sin(th),np.cos(th),np.zeros_like(th)])
        t2=np.column_stack([-z*np.cos(th),-z*np.sin(th),a])
        x=local@B3.T
        T=np.stack([t1@B3.T,t2@B3.T],axis=1)
        return x,T
    if family=='torus':
        u,v=p[:,0],p[:,1]
        cv,sv,cu,su=np.cos(v),np.sin(v),np.cos(u),np.sin(u)
        radial=R_MAJOR+R_MINOR*cv
        local=np.column_stack([radial*cu,radial*su,R_MINOR*sv])
        # Coordinate derivative directions normalized to length 1.
        t1=np.column_stack([-su,cu,np.zeros_like(u)])
        t2=np.column_stack([-sv*cu,-sv*su,cv])
        return local@B3.T, np.stack([t1@B3.T,t2@B3.T],axis=1)
    raise ValueError(family)

def family_neighbor_chords(family,params,x,T):
    # Keep the original four offset magnitudes. For 2D, cover both tangent axes.
    n,d,_=T.shape
    neighbors=[]
    for j in range(d):
        for delta in OFFSET:
            if family=='ellipse':
                th=params[:,0]
                speed=np.sqrt((A_ELL*np.sin(th))**2+(B_ELL*np.cos(th))**2)
                shifted=params.copy(); shifted[:,0]+=delta/speed
                x2,_=family_points_and_tangent(family,shifted)
            elif family=='sphere':
                # Exact sphere great-circle step of arc length |delta|.
                x2=np.cos(delta/R_SPHERE)*x+(R_SPHERE*np.sin(delta/R_SPHERE))*T[:,j,:]
            elif family=='torus':
                u,v=params[:,0],params[:,1]
                speed=(R_MAJOR+R_MINOR*np.cos(v)) if j==0 else np.full(n,R_MINOR)
                shifted=params.copy(); shifted[:,j]+=delta/speed
                x2,_=family_points_and_tangent(family,shifted)
            vec=x2-x
            neighbors.append(vec/(np.linalg.norm(vec,axis=1,keepdims=True)+1e-12))
    return np.stack(neighbors,axis=1)

def generalized_diagnostics(Q,T,x,params,family,rng):
    # Same original proxies; projector error normalized by sqrt(true tangent dim).
    Q=0.5*(Q+np.swapaxes(Q,-1,-2))
    eig,V=np.linalg.eigh(Q)
    n,d,Dlocal=T.shape
    P_hat=V[:,:,:d] @ np.swapaxes(V[:,:,:d],-1,-2)
    P_true=np.swapaxes(T,-1,-2)@T
    overlap=np.einsum('nij,nij->n',P_hat,P_true)
    err=np.sqrt(np.maximum(0.0,2.0*(d-overlap)/d))
    defect=np.linalg.norm(Q@Q-Q,axis=(1,2))/(np.linalg.norm(Q,axis=(1,2))+1e-12)
    nb=family_neighbor_chords(family,params,x,T)
    rand=rng.normal(size=(n,16,Dlocal))
    rand/=np.linalg.norm(rand,axis=2,keepdims=True)+1e-12
    nb_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,nb)**2,axis=-1),axis=1)
    rand_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,rand)**2,axis=-1),axis=1)
    proxy=np.minimum(1.0,nb_energy/(rand_energy+1e-12))
    soft_dim=np.sum(1-np.clip(eig,0,1),axis=1)
    return err,defect,proxy,soft_dim

In [ ]:
def circle_point_data(split):
    theta=np.random.default_rng(SEED+(1 if split=='fit' else 2)).uniform(0,2*np.pi,
        N_FIT if split=='fit' else N_TEST)
    coords=np.cos(theta)[:,None]*B[:,0][None,:]+np.sin(theta)[:,None]*B[:,1][None,:]
    tangent=(-np.sin(theta)[:,None]*B[:,0][None,:]+np.cos(theta)[:,None]*B[:,1][None,:])[:,None,:]
    indices=np.arange(N_FIT if split=='fit' else N_TEST)+(0 if split=='fit' else N_FIT)
    return indices,coords,tangent,theta[:,None]

def family_data(fam,split):
    if fam=='circle':return circle_point_data(split)
    assert fam in FAMILIES
    fi=FAMILIES.index(fam)
    seed=SEED+300+10*fi+(0 if split=='fit' else 1)
    n=N_FIT if split=='fit' else N_TEST
    p=sample_family_params(fam,n,np.random.default_rng(seed))
    x,T=family_points_and_tangent(fam,p)
    indices=np.arange(n)+(0 if split=='fit' else N_FIT)
    return indices,x,T,p

def checkpoint_for(fam):
    if fam=='circle':return DATA_DIR/'synthetic_eps_network.pt'
    return DATA_DIR/f'{fam}_eps_R8_seed{SEED}_steps2500_v1.pt'

def get_model(fam):
    p=checkpoint_for(fam)
    assert p.exists(), f'Checkpoint missing: {p}. Run extended synthetic notebook first.'
    net=EpsNetwork(D).to(DEVICE)
    state=torch.load(p,map_location=DEVICE,weights_only=True)
    net.load_state_dict(state); net.eval()
    for v in net.parameters():v.requires_grad_(False)
    return net

@torch.no_grad()
def q_batches(model,x,sigma,h=0.001):
    X=np.asarray(x,dtype=np.float32)
    cols=[]
    for j in range(D):
        plus=X.copy();minus=X.copy()
        plus[:,j]+=h;minus[:,j]-=h
        y1=model(torch.from_numpy(plus).to(DEVICE),float(sigma)).cpu().numpy()
        y2=model(torch.from_numpy(minus).to(DEVICE),float(sigma)).cpu().numpy()
        cols.append((y1-y2)/(2*h))
    Q=float(sigma)*np.stack(cols,axis=-1)
    return 0.5*(Q+np.swapaxes(Q,-1,-2))

## 3 · Validate existing evidence and build missing LHSD shared-operator CSVs
Use the exact 8D trace of the official LHSD Soft-DoF **spectral head**; this does **not** reproduce official LHSD's full scale-selection algorithm. Never label our fixed-scale comparator "official LHSD end-to-end".

The saved previous `*_LHSD_shared_operator.csv` files are reused where available. For a missing family, we reconstruct the same fixed operator features using its checkpoint, existing 40 fit /100 test points, and its original synthetic neighbor-proxy CSV. Recomputed errors are audited against the old CSV (tolerance 0.02).

In [ ]:
def lhsd_soft_dof_from_Q(Q,soft_c=LHSD_SOFT_C,p=LHSD_SOFT_P):
    """Exact eigen-trace of *official* LHSD Soft_DoF spectral head, D=8 only.
    This is a mathematical equivalent to the full-probe expectation of Soft_DoF,
    NOT its published stochastic Lanczos implementation or full experiment.
    """
    lam=np.linalg.eigvalsh(0.5*(Q+np.swapaxes(Q,-1,-2)))
    t=np.maximum(lam,0.0)/soft_c
    return np.sum(1.0/(1.0+t**p),axis=-1)

# Exact numerical sanity check for a 2-tangent/6-normal spectrum.
assert np.isclose(lhsd_soft_dof_from_Q(np.diag([0,0,1,1,1,1,1,1])),2+6/(1+2**12))
print('Exact LHSD Soft_DoF spectral-head test: PASS')

In [ ]:
FAMILY_ORDER = ['circle','sphere','ellipse','torus']
SKIPPED = {}
AVAILABLE = []
for fam in REQUESTED_FAMILIES:
    original_path=DATA_DIR/f'{fam}_per_point_results.csv'
    if fam=='circle' and not original_path.exists(): original_path=DATA_DIR/'all_per_point_results.csv'
    if not original_path.exists() or not checkpoint_for(fam).exists():
        SKIPPED[fam]=f'Missing {original_path.name} or {checkpoint_for(fam).name}'
    else: AVAILABLE.append(fam)
if not AVAILABLE: raise FileNotFoundError('No family has both original CSV and pretrained checkpoint.')
print('Available manifolds:',AVAILABLE)
print('Skipped manifolds:',SKIPPED)

REQUIRED = {'model','split','index','sigma','tangent_error','neighbor_proxy','projector_defect','soft_dimension'}
def old_table(fam):
    path=DATA_DIR/f'{fam}_per_point_results.csv'
    if fam=='circle' and not path.exists():path=DATA_DIR/'all_per_point_results.csv'
    tab=pd.read_csv(path)
    assert REQUIRED <= set(tab.columns),(fam,REQUIRED-set(tab.columns))
    tab=tab[tab.model=='learned_score'].copy()
    # For early circle CSV versions, original test IDs overlapped fitting IDs; split is still disjoint.
    if tab.groupby('index').split.nunique().gt(1).any():
        fit_max=int(tab.loc[tab.split=='fit','index'].max())
        tab.loc[tab.split=='test','index'] += fit_max+1
    return tab

def verify_frame(fam,frame):
    assert frame['split'].isin(['fit','test']).all()
    assert not frame.duplicated(['split','index','sigma']).any()
    assert frame.groupby(['split','index']).sigma.nunique().eq(len(SIGMAS)).all(),fam
    assert frame.loc[frame.split=='fit','index'].nunique()==N_FIT,(fam,'unexpected fit count')
    assert frame.loc[frame.split=='test','index'].nunique()==100,(fam,'unexpected historical test count')
    assert np.all(np.isfinite(frame[['sigma','tangent_error','neighbor_proxy','projector_defect','soft_dimension']]))

EVIDENCE_FILES={}
for fam in AVAILABLE:
    file=BENCH_DIR/f'{fam}_LHSD_shared_operator.csv'
    if file.exists():
        frame=pd.read_csv(file)
        assert {'dim_error','dim_true'}<=set(frame), f'{file}: missing LHSD metrics'
        verify_frame(fam,frame)
        print(fam,'reusing old LHSD shared CSV:',file)
        EVIDENCE_FILES[fam]=file
        continue
    print(fam,'LHSD shared CSV absent; reconstructing with same frozen score network')
    oldframe=old_table(fam)
    net=get_model(fam)
    data=[]
    for spl in ['fit','test']:
        ids,X,T,params=family_data(fam,spl)
        dtrue=T.shape[1]
        for sig in SIGMAS:
            Q=q_batches(net,X,sig)
            eig,V=np.linalg.eigh(Q)
            P=V[:,:,:dtrue]@np.swapaxes(V[:,:,:dtrue],-1,-2)
            Pt=np.swapaxes(T,-1,-2)@T
            errs=np.linalg.norm(P-Pt,axis=(1,2))/np.sqrt(dtrue)
            dims=lhsd_soft_dof_from_Q(Q)
            defects=np.linalg.norm(Q@Q-Q,axis=(1,2))/(np.linalg.norm(Q,axis=(1,2))+1e-12)
            for i,ix in enumerate(ids):
                matches=oldframe[(oldframe.split==spl)&(oldframe['index']==ix)&np.isclose(oldframe.sigma,sig)]
                if len(matches)!=1:raise ValueError((fam,spl,int(ix),sig,len(matches)))
                olderr=float(matches.tangent_error.iloc[0])
                if abs(olderr-float(errs[i]))>.02:raise RuntimeError(f'{fam} saved geometry/score checkpoint mismatch at index {ix}, sigma {sig}')
                data.append(dict(model='learned_score',split=spl,index=int(ix),sigma=sig,
                    tangent_error=float(errs[i]),neighbor_proxy=float(matches.neighbor_proxy.iloc[0]),
                    projector_defect=float(defects[i]),soft_dimension=float(dims[i]),
                    dim_true=dtrue,dim_error=float(abs(dims[i]-dtrue)),
                    source='reconstructed_official_LHSD_SoftDoF_exact_D8_trace'))
    frame=pd.DataFrame(data)
    verify_frame(fam,frame)
    file=OUT/f'{fam}_LHSD_shared_operator_reconstructed.csv'
    frame.to_csv(file,index=False)
    EVIDENCE_FILES[fam]=file
    print('Saved validated reconstruction',file)
    del net
    if DEVICE=='cuda':torch.cuda.empty_cache()

# Experiment 1 — Spectral dimension versus tangent orientation

Use the **existing, previously viewed** synthetic benchmark first. Report per-sigma Spearman correlation and prevalence of low dimension error with high tangent error. This is exploratory evidence only.

Then Experiment 3 will compute fresh *estimated-rank* errors. We cannot infer estimated-rank tangent error from old CSVs alone, because their original tangent error selected eigenvectors using the **true** synthetic dimension.

In [ ]:
exp1=[]
fig,axs=plt.subplots(1,len(AVAILABLE),figsize=(5.1*len(AVAILABLE),4.2),squeeze=False)
for j,fam in enumerate(AVAILABLE):
    df=pd.read_csv(EVIDENCE_FILES[fam])
    old=df[df.split=='test'].copy()
    old['dimension_error']=abs(old.soft_dimension-old.dim_true)
    ax=axs[0,j]
    for sig,g in old.groupby('sigma'):
        ax.scatter(g.dimension_error,g.tangent_error,alpha=.48,s=15,label=f'σ={sig:g}')
    ax.axvline(DIM_GOOD_CUTOFF,color='gray',ls='--',lw=1)
    ax.axhline(TANGENT_BAD_CUTOFF,color='gray',ls='--',lw=1)
    ax.set(xlabel='LHSD spectral dimension absolute error',ylabel='Oracle-rank tangent projector error',title=fam+' (old, exploratory)')
    ax.legend(fontsize=8)
    for scope,g in [('pooled',old)]+[(f'sigma_{sig:g}',g) for sig,g in old.groupby('sigma')]:
        rho,p=spearmanr(g.dimension_error,g.tangent_error)
        exp1.append(dict(family=fam,subset=scope,n_points=g['index'].nunique(),n_rows=len(g),
            spearman_rho=rho,spearman_p=p,
            dim_good_orientation_bad_fraction=float(((g.dimension_error<=DIM_GOOD_CUTOFF)&(g.tangent_error>TANGENT_BAD_CUTOFF)).mean()),
            mean_dimension_MAE=float(g.dimension_error.mean()),mean_oracle_rank_tangent_error=float(g.tangent_error.mean()),
            test_reuse='PREVIOUSLY_EXAMINED_EXPLORATORY'))
fig.tight_layout();fig.savefig(OUT/'Fig1_Dimension_vs_Orientation_OLD.png',dpi=180,bbox_inches='tight');plt.show()
exp1_df=pd.DataFrame(exp1)
exp1_df.to_csv(OUT/'EXPERIMENT1_OLD_DESCRIPTIVES.csv',index=False)
display(exp1_df.round(4))

# Experiment 2 — Do existing uncertainty signals identify failures?

**Fair protocol:** rank *the same candidate estimates* by each observable score. For every old point, choose a candidate scale using the **frozen** noise-aware predictor. On that identical set of chosen estimates, compare projector defect, neighbor proxy, predicted tangent error, and conformal upper bound. This isolates *failure ranking* from *noise-scale selection*. Compute AURC, Spearman, AUROC when both classes exist, and true error at prespecified coverages.

**Important:** the previously viewed 100-point tests are **exploratory**. We repeat exactly this analysis on new points in Experiment 3. AUROC is undefined when a failure threshold produces no positive or no negative labels.

In [ ]:
FROZEN={}
def fit_old_frozen(fam):
    df=pd.read_csv(EVIDENCE_FILES[fam])
    train,calib,test=original_selector.create_splits(df,seed=SEED)
    model,name,cv=original_selector.choose_predictor(train,seed=SEED)
    train=original_selector.add_pred(train,model)
    calib=original_selector.add_pred(calib,model)
    test=original_selector.add_pred(test,model)
    floors,q,ncal,_=original_selector.calibrate(train,calib,alpha=.10)
    # Detect accidental mismatch against old generated predictions if file exists.
    prior=BENCH_DIR/f'{fam}_selected'/'test_points_with_predictions.csv'
    if prior.exists():
        recorded=pd.read_csv(prior)[['index','sigma','predicted_error']]
        comparison=test.merge(recorded,on=['index','sigma'],suffixes=('_new','_old'),validate='one_to_one')
        delta=float(abs(comparison.predicted_error_new-comparison.predicted_error_old).max())
        if delta>1e-6:raise AssertionError(f'{fam}: frozen predictor differs from old run: {delta}')
    FROZEN[fam]=dict(predictor=model,name=name,cv=cv,floors=floors,q=q,n_cal=ncal,
                     train_ids=sorted(train['index'].unique().tolist()),cal_ids=sorted(calib['index'].unique().tolist()))
    print(f'{fam}: frozen predictor={name}, q={q:.4f}, fit/cal={train["index"].nunique()}/{ncal}')
    return train,calib,test

def add_old_model_outputs(df,frozen):
    res=original_selector.add_pred(df,frozen['predictor'])
    res['predicted_upper_bound']=res.predicted_error+res.sigma.map(frozen['floors'])*frozen['q']
    return res

OLD_EVAL=[];FIT_DATA={};CAL_DATA={}
for fam in AVAILABLE:
    tr,ca,te=fit_old_frozen(fam)
    FIT_DATA[fam]=tr;CAL_DATA[fam]=ca
    te=add_old_model_outputs(te,FROZEN[fam])
    OLD_EVAL.append(te.assign(family=fam,source='OLD_PREVIOUSLY_EXAMINED'))
OLD_EVAL=pd.concat(OLD_EVAL,ignore_index=True)
OLD_EVAL.to_csv(OUT/'OLD_TEST_ALL_SCALES_FROZEN_PREDICTIONS.csv',index=False)

# Pure helpers reused unchanged when testing fresh points below.
def risk_curve(chosen,feature,coverages=SELECTION_COVERAGES,label='tangent_error'):
    z=chosen.sort_values([feature,'index'],kind='mergesort').reset_index(drop=True)
    rows=[]
    for c in coverages:
        k=max(1,int(np.ceil(c*len(z))))
        acc=z.head(k)
        rows.append(dict(feature=feature,coverage=k/len(z),n_accepted=k,
                         mean_true_error=float(acc[label].mean()),
                         failure_rate=float((acc[label]>TANGENT_BAD_CUTOFF).mean())))
    return rows

def evaluate_failure_ranking(chosen,family,source,features):
    rows=[];curves=[]
    for feature in features:
        y=chosen.tangent_error.to_numpy();scores=chosen[feature].to_numpy()
        rho,_=spearmanr(y,scores)
        failure=(y>TANGENT_BAD_CUTOFF).astype(int)
        auc=np.nan if np.unique(failure).size<2 else roc_auc_score(failure,scores)
        cr=risk_curve(chosen,feature)
        aurc=float(np.trapezoid([v['mean_true_error'] for v in cr],
                                [v['coverage'] for v in cr]))
        r80=next(r['mean_true_error'] for r in cr if np.isclose(r['coverage'],.8,atol=.02))
        rows.append(dict(family=family,source=source,feature=feature,n_points=len(chosen),
            spearman_rho=rho,failure_AUROC=auc,AURC_20_to_100pct=aurc,
            true_error_at_80pct_coverage=r80,failure_prevalence=float(failure.mean())))
        curves.extend([dict(family=family,source=source,**r) for r in cr])
    return rows,curves

old_ranking=[];old_curves=[]
DEFAULT_FEATS=['projector_defect','neighbor_proxy','predicted_error','predicted_upper_bound']
for fam,g in OLD_EVAL.groupby('family'):
    chosen=g.loc[g.groupby('index')['predicted_error'].idxmin()].copy()
    a,b=evaluate_failure_ranking(chosen,fam,'OLD_EXPLORATORY',DEFAULT_FEATS)
    old_ranking+=a;old_curves+=b
old_rank_df=pd.DataFrame(old_ranking);old_curves_df=pd.DataFrame(old_curves)
old_rank_df.to_csv(OUT/'EXPERIMENT2_OLD_RANKINGS.csv',index=False)
old_curves_df.to_csv(OUT/'EXPERIMENT2_OLD_RISK_COVERAGE.csv',index=False)
display(old_rank_df.round(4))
fig,axs=plt.subplots(1,len(AVAILABLE),figsize=(5.5*len(AVAILABLE),4),squeeze=False)
for j,fam in enumerate(AVAILABLE):
    ax=axs[0,j]
    for feat,g in old_curves_df[old_curves_df.family==fam].groupby('feature'):
        ax.plot(g.coverage,g.mean_true_error,'o-',label=feat)
    ax.set(xlabel='Fraction of estimates accepted',ylabel='Mean true tangent error',title=fam+' OLD exploratory')
    ax.grid(alpha=.2);ax.legend(fontsize=7)
fig.tight_layout();fig.savefig(OUT/'Fig2_Risk_Coverage_OLD.png',dpi=180,bbox_inches='tight');plt.show()

# Experiment 3 — Cross-scale denoiser-response diagnostic

**New hypothesis (not a guaranteed theorem):** For candidate tangent projector \(\hat P_\sigma\) and score denoiser at another noise scale \(\sigma'\), define

\[
D_{\sigma'}(x)=x-\sigma'\epsilon_\theta(x,\sigma'),\qquad
A_{\sigma'}=J D_{\sigma'}(x)=I-\sigma' J\epsilon_\theta(x,\sigma') .
\]

The epsilon network was trained under \(x_\sigma=x+\sigma\epsilon\). The operator is **not automatically a conservative Hessian**. We retain the *raw* Jacobian for the denoiser response; spectral eigenspaces use its symmetric part.

\[
R_T^2=\|(A_{\sigma'}-I)\hat P_\sigma\|_F^2,
\quad
R_N^2=\|A_{\sigma'}(I-\hat P_\sigma)\|_F^2,
\]
\[
U_\mathrm{response}^2=R_T^2+R_N^2=\|A_{\sigma'}-\hat P_\sigma\|_F^2.
\]

**No tangent ground-truth is used in the new U score.** The exact triangle inequality gives
\[\|\hat P-P_\mathrm{true}\|_F\le \|A_{\sigma'}-\hat P\|_F+\|A_{\sigma'}-P_\mathrm{true}\|_F .\]
The unknown final term **prevents any claimed true-error upper guarantee**. This experiment only tests whether the observable residual predicts real synthetic error.

Use a *different* evaluation scale via `RESPONSE_SCALE`; also record same-scale response and scale projector disagreement as ablations. All reference scales were fixed in Cell 2. Report results separately by noise scale to detect scale-induced ranking bias.

In [ ]:
# Analytic unit test: ideal manifold projection must give zero residual.
rng=np.random.default_rng(708)
Z,_=np.linalg.qr(rng.normal(size=(D,2)))
P=Z@Z.T;A=P.copy()
assert np.isclose(np.linalg.norm(A-P),0,atol=1e-12)
W=np.eye(D);theta=0.25
W[0,0]=math.cos(theta);W[0,1]=-math.sin(theta)
W[1,0]=math.sin(theta);W[1,1]=math.cos(theta)
Pwrong=W@P@W.T
assert np.linalg.norm(A-Pwrong)>0

def raw_q_batches(model,x,sigma,h=0.001):
    # Q_raw = sigma * J_epsilon, same 8-dimensional finite differencing as prior notebooks.
    X=np.asarray(x,dtype=np.float32)
    cols=[]
    with torch.no_grad():
        for j in range(D):
            plus=X.copy();minus=X.copy();plus[:,j]+=h;minus[:,j]-=h
            a=model(torch.from_numpy(plus).to(DEVICE),float(sigma)).cpu().numpy()
            b=model(torch.from_numpy(minus).to(DEVICE),float(sigma)).cpu().numpy()
            cols.append((a-b)/(2*h))
    return float(sigma)*np.stack(cols,axis=-1)

def make_projectors(Qsym,rank):
    vals,V=np.linalg.eigh(Qsym)
    result=np.zeros_like(Qsym)
    for i,r in enumerate(rank):
        r=int(r)
        if r>0:result[i]=V[i,:,:r]@V[i,:,:r].T
    return result, vals, V

def circle_chords(theta,x):
    offsets=np.array([-0.14,-0.08,0.08,0.14])
    neighbors=[]
    for delta in offsets:
        angled=theta[:,0]+delta
        x2=np.cos(angled)[:,None]*B[:,0]+np.sin(angled)[:,None]*B[:,1]
        z=x2-x;neighbors.append(z/(np.linalg.norm(z,axis=1,keepdims=True)+1e-12))
    return np.stack(neighbors,axis=1)

def geometry_neighborhood_proxy(fam,X,T,params,Q,sig,seed):
    # Reproduces original synthetic proxy (uses analytic parametrization; NOT a label-free real-data kNN).
    chords=circle_chords(params,X) if fam=='circle' else family_neighbor_chords(fam,params,X,T)
    rng=np.random.default_rng(seed)
    z=rng.normal(size=(len(X),16,D));z/=np.linalg.norm(z,axis=-1,keepdims=True)+1e-12
    e_chord=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,chords)**2,axis=-1),axis=1)
    e_rand=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,z)**2,axis=-1),axis=1)
    return np.minimum(1.0,e_chord/(e_rand+1e-12))

def measure_family_points(fam,net,ids,X,T,params,split_tag,proxy_from_old=None):
    # Computes all three candidate Q at each x and cross-noise denoiser response.
    n=len(X); dtrue=T.shape[1]
    Pt=np.swapaxes(T,-1,-2)@T
    Qraw={s:raw_q_batches(net,X,s) for s in SIGMAS}
    Qsym={s:.5*(Qraw[s]+np.swapaxes(Qraw[s],-1,-2)) for s in SIGMAS}
    out=[]
    P_est={}; P_oracle={}; dim_est={}; rank_est={}
    for s in SIGMAS:
        q=Qsym[s]
        lam,V=np.linalg.eigh(q)
        dof=lhsd_soft_dof_from_Q(q)
        rank=np.clip(np.rint(dof).astype(int),0,D)
        Phat,_,_=make_projectors(q,rank)
        Poracle=V[:,:,:dtrue]@np.swapaxes(V[:,:,:dtrue],-1,-2)
        P_est[s]=Phat;P_oracle[s]=Poracle;dim_est[s]=dof;rank_est[s]=rank
    for sig in SIGMAS:
        q=Qsym[s];qraw=Qraw[s]
        EVAL=RESPONSE_SCALE[sig]
        A=np.eye(D)[None,:,:]-Qraw[EVAL]
        P=P_est[sig]
        Poracle=P_oracle[sig]
        I=np.eye(D)[None,:,:]
        residual=A-P
        Rt=np.linalg.norm((A-I)@P,axis=(1,2))/np.sqrt(D)
        Rn=np.linalg.norm(A@(I-P),axis=(1,2))/np.sqrt(D)
        cross=np.sqrt(Rt**2+Rn**2)
        direct=np.linalg.norm(residual,axis=(1,2))/np.sqrt(D)
        assert np.allclose(cross,direct,atol=1e-6,rtol=1e-5)
        A_self=I-qraw
        response_self=np.linalg.norm(A_self-P,axis=(1,2))/np.sqrt(D)
        # To measure projector movement, impose the same candidate rank on the other scale.
        _,Vother=np.linalg.eigh(Qsym[EVAL])
        Pnext=np.zeros_like(P)
        for i,k in enumerate(rank_est[sig]):
            if k>0:Pnext[i]=Vother[i,:,:k]@Vother[i,:,:k].T
        proj_shift=np.linalg.norm(P-Pnext,axis=(1,2))/np.sqrt(D)
        anti=np.linalg.norm(qraw-np.swapaxes(qraw,-1,-2),axis=(1,2))/(np.linalg.norm(q,axis=(1,2))+1e-12)
        if proxy_from_old is None:
            proxy=geometry_neighborhood_proxy(fam,X,T,params,q,sig,SEED+13)
        else:
            src=proxy_from_old[np.isclose(proxy_from_old.sigma,sig)].set_index('index')
            proxy=np.array([float(src.loc[int(ix),'neighbor_proxy']) for ix in ids])
        defect=np.linalg.norm(q@q-q,axis=(1,2))/(np.linalg.norm(q,axis=(1,2))+1e-12)
        e_oracle=np.linalg.norm(Poracle-Pt,axis=(1,2))/np.sqrt(dtrue)
        e_rank=np.linalg.norm(P-Pt,axis=(1,2))/np.sqrt(dtrue)
        for i,ix in enumerate(ids):
            out.append(dict(family=fam,model='learned_score',split=split_tag,index=int(ix),
                sigma=float(sig),tangent_error=float(e_oracle[i]),estimated_rank_tangent_error=float(e_rank[i]),
                dim_true=int(dtrue),soft_dimension=float(dim_est[sig][i]),estimated_rank=int(rank_est[sig][i]),
                dim_error=float(abs(dim_est[sig][i]-dtrue)),
                projector_defect=float(defect[i]),neighbor_proxy=float(proxy[i]),
                response_cross=float(direct[i]),response_same=float(response_self[i]),
                response_tangent=float(Rt[i]),response_normal=float(Rn[i]),
                scale_projector_disagreement=float(proj_shift[i]),
                antisymmetry=float(anti[i]),reference_sigma=float(EVAL)))
    return pd.DataFrame(out)

# Boundary check for rank and response identities.
Qtoy=np.broadcast_to((np.eye(D)-P), (3,D,D)).copy()
assert make_projectors(Qtoy,np.array([2,2,2]))[0].shape==(3,D,D)
print('Cross-scale response identity and synthetic projector tests: PASS')

## 3A · Reconstruct old FIT points, then generate **fresh** audit points

For each manifold, use the original 40 fitting points for the original 20/20 split. The audit set is generated from fresh fixed RNG seeds and has never appeared in your earlier CSVs. We do not retrain the score network. **Experiments 1 and 2 on the old test remain exploratory; only the new-point analysis is a prospective point-level test.**

The original synthetic `neighbor_proxy` was created using manifold-parametrized chords, which require privileged geometric knowledge on sphere and other families. We preserve it to compare like-with-like, but do **not** call that proxy fully label-free. `response_cross`, `response_tangent`, `response_normal`, and `scale_projector_disagreement` do not use the true tangent basis for their computation.

In [ ]:
def generate_fresh_points(fam,n):
    rng=np.random.default_rng(FRESH_POINT_SEED+97*FAMILY_ORDER.index(fam))
    if fam=='circle':
        theta=rng.uniform(0,2*np.pi,size=(n,1))
        X=np.cos(theta)*B[:,0]+np.sin(theta)*B[:,1]
        T=(-np.sin(theta)*B[:,0]+np.cos(theta)*B[:,1])[:,None,:]
        params=theta
    else:
        params=sample_family_params(fam,n,rng)
        X,T=family_points_and_tangent(fam,params)
    ids=np.arange(n)+100000+1000*FAMILY_ORDER.index(fam)
    return ids,X,T,params

SAVED={}
all_data=[]
for fam in AVAILABLE:
    file=OUT/f'{fam}_RESPONSE_ALL_CANDIDATES.csv'
    if file.exists():
        tab=pd.read_csv(file)
        assert {'fit','fresh_test'}==set(tab.split),f'{fam}: wrong cached split'
        assert tab[tab.split=='fresh_test']['index'].nunique()==N_FRESH_AUDIT, 'Clear stale cache after changing N_FRESH_AUDIT'
        for s,r in RESPONSE_SCALE.items():
            t=tab[np.isclose(tab.sigma,s)]
            assert np.allclose(t.reference_sigma,r), 'Cached reference noise grid changed; remove response CSV to recompute'
        print(fam,'reused previously computed diagnostic CSV',file)
    else:
        print('Computing new cross-scale score responses:',fam)
        net=get_model(fam)
        ids0,X0,T0,p0=family_data(fam,'fit')
        fit_old=pd.read_csv(EVIDENCE_FILES[fam])
        fit_old=fit_old[fit_old.split=='fit'].copy()
        fit=measure_family_points(fam,net,ids0,X0,T0,p0,'fit',proxy_from_old=fit_old)
        # Verify score-network, sample, sigma, and test-label alignment.
        joined=fit.merge(fit_old[['index','sigma','tangent_error','soft_dimension']],on=['index','sigma'],suffixes=('_now','_old'),validate='one_to_one')
        max_err=float(abs(joined.tangent_error_now-joined.tangent_error_old).max())
        max_dim=float(abs(joined.soft_dimension_now-joined.soft_dimension_old).max())
        assert max_err<0.02,(fam,'checkpoint/embedding mismatch',max_err)
        assert max_dim<0.03,(fam,'spectral head mismatch',max_dim)
        ids1,X1,T1,p1=generate_fresh_points(fam,N_FRESH_AUDIT)
        fresh=measure_family_points(fam,net,ids1,X1,T1,p1,'fresh_test')
        tab=pd.concat([fit,fresh],ignore_index=True)
        tab.to_csv(file,index=False)
        print(f'{fam}: FIT agreement errors: true tangent {max_err:.6f}, LHSD DoF {max_dim:.6f}')
        del net
        if DEVICE=='cuda':torch.cuda.empty_cache()
    SAVED[fam]=file
    all_data.append(tab)
RESPONSE_ALL=pd.concat(all_data,ignore_index=True)
assert not RESPONSE_ALL.duplicated(['family','split','index','sigma']).any()
assert RESPONSE_ALL.groupby(['family','split','index']).sigma.nunique().eq(len(SIGMAS)).all()
RESPONSE_ALL.to_csv(OUT/'RESPONSE_ALL_FAMILIES.csv',index=False)
display(RESPONSE_ALL.groupby(['family','split']).agg(points=('index','nunique'),rows=('index','size'),
    mean_oracle_rank_error=('tangent_error','mean'),mean_estimated_rank_error=('estimated_rank_tangent_error','mean')).round(4))

## 3B · Experiment 1 confirmation on fresh points — **estimated rank vs oracle rank**

The estimated rank is `round(LHSD SoftDoF)` clipped to `[0,D]` solely to produce an integer-rank projector. **No true intrinsic rank enters this estimator**. The synthetic rank enters only the evaluation label and the oracle-rank comparator.

In [ ]:
new1=[]
fig,axs=plt.subplots(2,len(AVAILABLE),figsize=(5*len(AVAILABLE),8),squeeze=False)
for j,fam in enumerate(AVAILABLE):
    g=RESPONSE_ALL[(RESPONSE_ALL.family==fam)&(RESPONSE_ALL.split=='fresh_test')]
    for k,label in enumerate(['tangent_error','estimated_rank_tangent_error']):
        ax=axs[k,j]
        for sig,part in g.groupby('sigma'):
            ax.scatter(part.dim_error,part[label],s=13,alpha=.42,label=f'σ={sig:g}')
        ax.axvline(DIM_GOOD_CUTOFF,color='gray',ls='--')
        ax.axhline(TANGENT_BAD_CUTOFF,color='gray',ls='--')
        ax.set(xlabel='LHSD dimension absolute error',ylabel=label.replace('_',' '),title=f'{fam}: {"oracle rank" if k==0 else "estimated rank"}')
        if k==0:ax.legend(fontsize=8)
        rho,p=spearmanr(g.dim_error,g[label])
        new1.append(dict(family=fam,metric=label,n_points=g['index'].nunique(),spearman_rho=rho,
            fraction_dim_good_orientation_bad=float(((g.dim_error<=DIM_GOOD_CUTOFF)&(g[label]>TANGENT_BAD_CUTOFF)).mean()),
            mean_error=float(g[label].mean()),mean_dimension_error=float(g.dim_error.mean()),
            rank_match_rate=float((g.estimated_rank==g.dim_true).mean())))
fig.tight_layout();fig.savefig(OUT/'Fig1_Dimension_vs_Orientation_FRESH.png',dpi=180,bbox_inches='tight');plt.show()
new1_df=pd.DataFrame(new1)
new1_df.to_csv(OUT/'EXPERIMENT1_FRESH.csv',index=False)
display(new1_df.round(4))

## 3C · Experiment 2 confirmation on fresh points, plus the new response feature

All features rank exactly the same **frozen-selector-chosen estimate** for each point. We separately report estimated-rank and oracle-rank failures. Pre-existing test analyses above are exploratory; this is new-point evaluation using previously fixed failure threshold and candidate scale choices.

In [ ]:
NEW_PRED=[]
for fam in AVAILABLE:
    sub=RESPONSE_ALL[(RESPONSE_ALL.family==fam)&(RESPONSE_ALL.split=='fresh_test')].copy()
    sub=add_old_model_outputs(sub,FROZEN[fam])
    # Response scores across candidate scales can have different distributions, so
    # this cell compares uncertainty rankings among points AFTER a frozen scale choice.
    NEW_PRED.append(sub)
NEW_PRED=pd.concat(NEW_PRED,ignore_index=True)
NEW_PRED.to_csv(OUT/'FRESH_ALL_SCALES_FROZEN_PREDICTIONS.csv',index=False)
NEW_CHOSEN=NEW_PRED.loc[NEW_PRED.groupby(['family','index'])['predicted_error'].idxmin()].copy()
# Check the choice is exactly one row per family+point.
assert not NEW_CHOSEN.duplicated(['family','index']).any()
NEW_CHOSEN.to_csv(OUT/'FRESH_FROZEN_SELECTED_CANDIDATES.csv',index=False)
FEATURES_EXT=['projector_defect','neighbor_proxy','predicted_error','predicted_upper_bound',
              'response_cross','response_same','scale_projector_disagreement','antisymmetry']
rows2=[];curves2=[]
for fam,g in NEW_CHOSEN.groupby('family'):
    for outcome in ['tangent_error','estimated_rank_tangent_error']:
        h=g.rename(columns={outcome:'TARGET_TMP'}).copy()
        for feat in FEATURES_EXT:
            score=h[feat].to_numpy();target=h.TARGET_TMP.to_numpy()
            rho,_=spearmanr(score,target)
            fail=(target>TANGENT_BAD_CUTOFF).astype(int)
            auc=np.nan if len(np.unique(fail))<2 else roc_auc_score(fail,score)
            z=h.sort_values([feat,'index'],kind='mergesort')
            cr=[]
            for c in SELECTION_COVERAGES:
                accepted=z.iloc[:max(1,int(np.ceil(c*len(z))))]
                cr.append(dict(coverage=len(accepted)/len(z),mean_true_error=float(accepted.TARGET_TMP.mean()),
                    failure_rate=float((accepted.TARGET_TMP>TANGENT_BAD_CUTOFF).mean())))
                curves2.append(dict(family=fam,outcome=outcome,feature=feat,**cr[-1]))
            rows2.append(dict(family=fam,outcome=outcome,feature=feat,n_points=len(g),
                spearman_rho=rho,failure_AUROC=auc,failure_prevalence=float(fail.mean()),
                AURC_20_to_100pct=float(np.trapezoid([r['mean_true_error'] for r in cr],[r['coverage'] for r in cr])),
                error_at_80pct_coverage=float(cr[3]['mean_true_error'])))
rank_new=pd.DataFrame(rows2)
curve_new=pd.DataFrame(curves2)
rank_new.to_csv(OUT/'EXPERIMENT2_FRESH_RANKINGS.csv',index=False)
curve_new.to_csv(OUT/'EXPERIMENT2_FRESH_RISK_COVERAGE.csv',index=False)
# Paired bootstrap over the SAME selected points, comparing AURC of the new
# diagnostic against preexisting uncertainty scores. No noise-scale rows treated as iid.
def computed_aurc(frame,feature,outcome):
    order=frame.sort_values(feature,kind='mergesort')
    curves=[]
    for c in SELECTION_COVERAGES:
        k=max(1,int(np.ceil(c*len(order))))
        curves.append((k/len(order),float(order.iloc[:k][outcome].mean())))
    return float(np.trapezoid([y for x,y in curves],[x for x,y in curves]))
paired_aurc=[]
for fam,g in NEW_CHOSEN.groupby('family'):
    for outcome in ['tangent_error','estimated_rank_tangent_error']:
        for base in ['predicted_error','neighbor_proxy','projector_defect']:
            estimate=computed_aurc(g,base,outcome)-computed_aurc(g,'response_cross',outcome)
            rnd=np.random.default_rng(3121+FAMILY_ORDER.index(fam))
            ids=np.arange(len(g))
            bvals=[]
            for _ in range(N_BOOT):
                sample=g.iloc[rnd.choice(ids,size=len(ids),replace=True)]
                bvals.append(computed_aurc(sample,base,outcome)-computed_aurc(sample,'response_cross',outcome))
            ci=np.quantile(bvals,[.025,.975])
            paired_aurc.append(dict(family=fam,outcome=outcome,baseline=base,n_points=len(g),
                mean_AURC_baseline_minus_response=estimate,CI95_low=ci[0],CI95_high=ci[1],
                conclusion=('response better' if ci[0]>0 else 'response worse' if ci[1]<0 else 'inconclusive')))
aurc_ci_df=pd.DataFrame(paired_aurc)
aurc_ci_df.to_csv(OUT/'EXPERIMENT2_PAIRED_AURC_CI95.csv',index=False)
display(aurc_ci_df.round(5))
display(rank_new[(rank_new.outcome=='tangent_error')].sort_values(['family','AURC_20_to_100pct']).round(4))
fig,axs=plt.subplots(1,len(AVAILABLE),figsize=(5.6*len(AVAILABLE),4),squeeze=False)
for j,fam in enumerate(AVAILABLE):
    ax=axs[0,j]
    selected_features=['projector_defect','neighbor_proxy','predicted_error','response_cross','scale_projector_disagreement']
    for feat in selected_features:
        zz=curve_new[(curve_new.family==fam)&(curve_new.outcome=='tangent_error')&(curve_new.feature==feat)]
        ax.plot(zz.coverage,zz.mean_true_error,'o-',label=feat)
    ax.set(xlabel='Coverage',ylabel='Mean actual tangent error',title=f'{fam}: FRESH')
    ax.grid(alpha=.2);ax.legend(fontsize=7)
fig.tight_layout();fig.savefig(OUT/'Fig2_Risk_Coverage_FRESH.png',dpi=180,bbox_inches='tight');plt.show()

## 3D · Does force-response add **independent** predictive information?

One critical ablation: fit **two equal-complexity ridge regressors** on the *same original 20 FIT points* (all three scales), with hyperparameters fixed in advance:

- Baseline: `sigma`, neighbor proxy, defect, LHSD dimension.
- Baseline + `response_cross`.

No response weights or thresholds are tuned on new points. Evaluate predictive MAE and Spearman on fresh points, paired bootstrap by independent **point**, not by noise-scale row. Also examine per-sigma correlations (different reference scales can bias scores). A positive paired CI supports incremental predictive information in this experimental setting; a negative result argues against adding this feature.

In [ ]:
from sklearn.metrics import mean_absolute_error
abl=[];abl_preds=[]
for fam in AVAILABLE:
    z=RESPONSE_ALL[RESPONSE_ALL.family==fam].copy()
    trainids=set(FROZEN[fam]['train_ids'])
    tr=z[(z.split=='fit')&(z['index'].isin(trainids))].copy()
    te=z[z.split=='fresh_test'].copy()
    keys0=['sigma','neighbor_proxy','projector_defect','soft_dimension']
    keys1=keys0+['response_cross']
    assert tr['index'].nunique()==20, (fam,'Original FIT must be 20')
    m0=make_pipeline(StandardScaler(),Ridge(alpha=3.0))
    m1=make_pipeline(StandardScaler(),Ridge(alpha=3.0))
    m0.fit(tr[keys0],tr.tangent_error)
    m1.fit(tr[keys1],tr.tangent_error)
    te['pred_base']=m0.predict(te[keys0]);te['pred_plus_response']=m1.predict(te[keys1])
    te['abs_err_base']=abs(te.pred_base-te.tangent_error)
    te['abs_err_plus']=abs(te.pred_plus_response-te.tangent_error)
    te['paired_improvement']=te.abs_err_base-te.abs_err_plus
    # independent-point bootstrap; all three correlated sigmas travel together
    diff=te.groupby('index').paired_improvement.mean().to_numpy()
    rng=np.random.default_rng(808+FAMILY_ORDER.index(fam))
    means=np.mean(diff[rng.integers(0,len(diff),size=(N_BOOT,len(diff)))],axis=1)
    ci=np.quantile(means,[.025,.975])
    abl.append(dict(family=fam,n_points=te['index'].nunique(),
        baseline_ridge_MAE=float(te.abs_err_base.mean()),
        plus_response_ridge_MAE=float(te.abs_err_plus.mean()),
        mean_MAE_improvement=float(diff.mean()),CI95_low=float(ci[0]),CI95_high=float(ci[1]),
        added_diagnostic_supported_on_these_points=bool(ci[0]>0)))
    abl_preds.append(te)
abl_df=pd.DataFrame(abl)
abl_df.to_csv(OUT/'EXPERIMENT3_INCREMENTAL_ABLATION.csv',index=False)
abl_preds=pd.concat(abl_preds,ignore_index=True)
abl_preds.to_csv(OUT/'EXPERIMENT3_RIDGE_FRESH_PREDICTIONS.csv',index=False)
display(abl_df.round(5))
fig,axs=plt.subplots(1,len(AVAILABLE),figsize=(5.8*len(AVAILABLE),4),squeeze=False)
for j,fam in enumerate(AVAILABLE):
    ax=axs[0,j]
    zz=RESPONSE_ALL[(RESPONSE_ALL.family==fam)&(RESPONSE_ALL.split=='fresh_test')]
    for s,g in zz.groupby('sigma'):
        ax.scatter(g.response_cross,g.tangent_error,alpha=.4,s=16,label=f'sigma={s:g}')
    rho,_=spearmanr(zz.response_cross,zz.tangent_error)
    ax.set(xlabel='Cross-scale denoiser response residual / sqrt(D)',ylabel='True oracle-rank tangent error',
           title=f'{fam} — pooled rho={rho:.3f}')
    ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'Fig3_Response_vs_Tangent_Error_FRESH.png',dpi=180,bbox_inches='tight');plt.show()
# Per-sigma correlation: do NOT hide cross-scale evaluation bias.
per_sigma=[]
for (fam,sig),g in RESPONSE_ALL[RESPONSE_ALL.split=='fresh_test'].groupby(['family','sigma']):
    for feature in ['response_cross','response_same','scale_projector_disagreement','projector_defect','neighbor_proxy']:
        rho,p=spearmanr(g[feature],g.tangent_error)
        per_sigma.append(dict(family=fam,sigma=sig,feature=feature,spearman_rho=rho,spearman_p=p,n=len(g)))
per_sigma_df=pd.DataFrame(per_sigma)
per_sigma_df.to_csv(OUT/'EXPERIMENT3_PER_SIGMA_CORRELATIONS.csv',index=False)
display(per_sigma_df.pivot_table(index=['family','sigma'],columns='feature',values='spearman_rho').round(3))

## 3E · Honest mathematical audit and proposal-ready evidence

- `response_cross` is **observable** but is not a bound on the unknown true tangent error.
- `response_same` may be circular: both `A` and `P` come from the same model at the same scale.
- `neighbor_proxy` uses synthetic analytically generated neighborhood chords; do not claim fully real-world label-free neighbor acquisition without a separate kNN experiment.
- One model checkpoint per family does not establish model-seed generalization.
- Soft-DoF is the exact-trace implementation of an official LHSD **spectral head**, not its complete published scale-selection method.
- Our synthetic calibration predictor is supervised using synthetic tangent labels on FIT/CAL; it cannot give a validated real-MNIST true-error guarantee.
- **If ridge+response does not outperform baseline**, do not claim novelty of the diagnostic. The next research question becomes why self-consistent score physics fails to detect tangent failure.

All exported numbers below are computed from artifacts; no fabricated model outcomes.

In [ ]:
# Programmatic evidence report: summarize numerical outcomes without inventing positive results.
protocol=dict(seed=SEED,fresh_point_seed=FRESH_POINT_SEED,n_fresh_points=N_FRESH_AUDIT,
              sigmas=SIGMAS,response_scale=RESPONSE_SCALE,device=DEVICE,
              available_families=AVAILABLE,skipped=SKIPPED,
              historical_dataset_examined_before_this_notebook=True,
              new_points_independent_of_historical_points=True,
              not_independent_score_model_training_seeds=True,
              rank_policy='round(LHSD SoftDoF), clip to [0,D], no true rank used in method',
              tangent_label_definition='Frobenius projector error normalized by sqrt(true dimension)',
              proxy_privilege='synthetic neighbor chords generated using analytic manifold parametrization',
              theory='residual <=/=> true error without bounding denoiser geometry bias')
(OUT/'EXPERIMENT_PROTOCOL.json').write_text(json.dumps(protocol,indent=2))

outlines=[
'# Proposal evidence — geometry reliability experiments 1–3','',
'## Scope',
'Reuses the pretrained R8 epsilon-model checkpoints, the original LHSD SoftDoF spectral head, and the same 20 FIT/20 CAL noise-aware predictor. No new score models were trained.',
'Previously examined results are exploratory; fresh point-level results are separately evaluated. Both oracle-rank and estimated-rank metrics are identified.',
'', '## Experiment 1 — dimension vs tangent orientation'
]
for _,r in new1_df.iterrows():
    outlines.append(f"- **{r.family}, {r.metric}**: dimension MAE {r.mean_dimension_error:.5f}; tangent error {r.mean_error:.5f}; Spearman rho {r.spearman_rho:.3f}; accurate-dimension/high-tangent-error fraction {r.fraction_dim_good_orientation_bad:.3f}; estimated-rank matches true rank {r.rank_match_rate:.3f}.")
outlines+=['','## Experiment 2 — new-point failure ranking (oracle rank)']
for _,r in rank_new[(rank_new.outcome=='tangent_error') & (rank_new.feature.isin(['predicted_error','response_cross','projector_defect']))].iterrows():
    outlines.append(f'- {r.family} / {r.feature}: Spearman rho={r.spearman_rho:.3f}, AURC={r.AURC_20_to_100pct:.5f}, AUROC={r.failure_AUROC if np.isfinite(r.failure_AUROC) else "undefined: no positive/negative failures"}.')
outlines+=['','## Experiment 3 — incremental diagnostic ablation']
for _,r in abl_df.iterrows():
    verdict='positive paired CI' if r.CI95_low>0 else ('negative paired CI' if r.CI95_high<0 else 'inconclusive paired CI')
    outlines.append(f'- {r.family}: baseline MAE={r.baseline_ridge_MAE:.5f}; +response MAE={r.plus_response_ridge_MAE:.5f}; improvement={r.mean_MAE_improvement:.5f}; 95% paired CI=[{r.CI95_low:.5f}, {r.CI95_high:.5f}]; {verdict}.')
outlines+=['','## Limitations',
'- One score-model checkpoint per family; results do not establish training-seed generalization.',
'- Fixed-scale LHSD-derived comparison is not official LHSD end-to-end.',
'- Current synthetic neighbor proxy uses analytic manifold parametrization and cannot be presented as entirely label-free for MNIST.',
'- Score-denoiser response consistency is a proxy; its difference from the true projector is unknown and not covered by a theorem.',
'- Historical tests have been examined. New samples are independent points but the hypothesis was informed by previous data.',
'- A formal true-tangent-error bound on real MNIST is NOT established.',
'', '## Figures',
'- Fig1_Dimension_vs_Orientation_FRESH.png',
'- Fig2_Risk_Coverage_FRESH.png',
'- Fig3_Response_vs_Tangent_Error_FRESH.png',
'']
(OUT/'PROPOSAL_EVIDENCE.md').write_text('\n'.join(outlines))
print('Evidence file:',OUT/'PROPOSAL_EVIDENCE.md')
print('Output files:',sorted(p.name for p in OUT.glob('*')))

## 4 · Export complete results

**Key exports:** `PROPOSAL_EVIDENCE.md`, `EXPERIMENT1_FRESH.csv`, `EXPERIMENT2_FRESH_RANKINGS.csv`, `EXPERIMENT3_INCREMENTAL_ABLATION.csv`, and the three `Fig*_FRESH.png` figures.

**Proposal success criterion:** reliable, orientation-sensitive diagnostics should outperform pre-existing scores **on new points**, with paired uncertainty intervals and consistent directions of effect across multiple families. If it fails on sphere/torus, that is an important limitation—not a license to change thresholds after inspecting the audit data.

**Next external-validity step:** independently retrain score networks with multiple seeds; then test a realistic unsupervised kNN proxy and real MNIST geometric consistency. Not required to run this notebook.

In [ ]:
archive=shutil.make_archive(str(OUT.parent/'Orientation_Reliability_Experiments_1_2_3'),'zip',root_dir=str(OUT))
print('Saved complete archive:',archive)
try:
    from google.colab import files
    # Optional download. Comment out on slow browser connections.
    files.download(archive)
except ImportError:
    pass